In [2]:
"""Outlook Email Extractor - Optimized & Focused Version
Chỉ extract attachment từ email CÓ file đính kèm (.xlsx hoặc .csv)
Bỏ qua email không có attachment
Tách biệt hoàn toàn: 
- Raw Excel lấy toàn bộ email (không phụ thuộc keyword)
- Attachment extract chỉ dựa trên SUBJECT_KEYWORDS + CÓ attachment
"""

from __future__ import annotations

import win32com.client as win32
import re
from pathlib import Path
import polars as pl
from datetime import date
from typing import List, Dict, Any

# ====================== CONFIG ======================
ACCOUNT_NAME = "huuchinh.nguyen@concentrix.com"
SAVE_RAW_EXCEL = True
RAW_EXCEL_FILE = Path("outlook_emails_raw.xlsx")
EXTRACT_ATTACHMENTS = True
ATTACHMENT_BASE = Path(
    r"C:\Users\ADMIN\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\INPUT_SCHEDULE_TEAM_MAIL"
)
SUBJECT_KEYWORDS = [
    "NL Chat Global OU Projections" #"LG Chat (Globally) Schedules Projections", 
]
PIVOT_FOLDERS = {"RTA team", "TL"}
BATCH_PRINT_EVERY = 2000
MAX_BODY_PREVIEW = 3000
FILTER_FROM_DATE: date | None = date(2026, 3, 1)  # None = không lọc ngày

def sanitize_folder_name(name: str) -> str:
    name = re.sub(r'[\\/:*?"<>|]', "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name[:150] or "No_Subject"


def has_relevant_subject(subject: str) -> bool:
    if not subject:
        return False
    subject_lower = subject.lower()
    return any(kw.lower() in subject_lower for kw in SUBJECT_KEYWORDS)


def has_valid_attachment(item) -> bool:
    if not hasattr(item, "Attachments") or item.Attachments.Count == 0:
        return False
    for att in item.Attachments:
        if att.FileName.lower().endswith((".xlsx", ".csv")):
            return True
    return False


def save_attachments(item, subject: str) -> bool:
    if not has_valid_attachment(item):
        return False

    folder_name = sanitize_folder_name(subject)
    save_path = ATTACHMENT_BASE / folder_name
    save_path.mkdir(parents=True, exist_ok=True)

    saved_any = False
    for att in item.Attachments:
        filename = att.FileName.lower()
        if filename.endswith((".xlsx", ".csv")):
            full_path = save_path / att.FileName
            try:
                att.SaveAsFile(str(full_path))
                print(f"   ✓ {folder_name}\\{att.FileName}")
                saved_any = True
            except Exception as e:
                print(f"   ✗ Failed {att.FileName}: {e}")
    return saved_any


def get_namespace():
    app = win32.Dispatch("Outlook.Application")
    return app.GetNamespace("MAPI")


def normalize_datetime(dt):
    return dt.replace(tzinfo=None) if dt else None


def get_full_path(folder) -> str:
    if not hasattr(folder, "Name"):
        return "[Unknown]"
    parts = [folder.Name]
    current = folder
    while True:
        try:
            parent = current.Parent
            if not hasattr(parent, "Name"):
                break
            parts.append(parent.Name)
            current = parent
        except AttributeError:
            break
    return " → ".join(reversed(parts)) if parts else "[Root]"


def collect_emails_recursive(folder, results: List[Dict[str, Any]]):
    if not hasattr(folder, "Items") or not hasattr(folder, "Name"):
        return

    items = folder.Items

    for i, item in enumerate(items, 1):
        if i % BATCH_PRINT_EVERY == 0:
            print(f"Scanned {i:,} items in {folder.Name}...")

        if not (hasattr(item, "Subject") and hasattr(item, "ReceivedTime")):
            continue

        received = normalize_datetime(item.ReceivedTime)
        if received is None:
            continue

        if FILTER_FROM_DATE and received.date() < FILTER_FROM_DATE:
            continue

        subject = item.Subject or "[No Subject]"
        if EXTRACT_ATTACHMENTS and has_valid_attachment(item) and has_relevant_subject(subject):
            save_attachments(item, subject)
        results.append({
            "Folder": folder.Name,
            "FullPath": get_full_path(folder),
            "Subject": subject,
            "Received": received,
            "Date": received.date(),
            "Sender": item.SenderName or item.SenderEmailAddress or "[Unknown]",
            "BodyPreview": (getattr(item, "Body", "") or "")[:MAX_BODY_PREVIEW],
        })

    if hasattr(folder, "Folders"):
        for subfolder in folder.Folders:
            collect_emails_recursive(subfolder, results)


def create_pivot(df: pl.DataFrame) -> pl.DataFrame:
    df_pivot = df.filter(pl.col("Folder").is_in(PIVOT_FOLDERS))
    pivot = (
        df_pivot.group_by("Folder", "Date")
        .agg(pl.len().alias("Email Count"))
        .sort("Date", descending=True)
        .pivot(index="Folder", columns="Date", values="Email Count", aggregate_function="sum")
        .fill_null(0)
    )
    return pivot


def main():
    ATTACHMENT_BASE.mkdir(parents=True, exist_ok=True)

    ns = get_namespace()
    account = ns.Folders(ACCOUNT_NAME)
    all_emails: List[Dict[str, Any]] = []
    collect_emails_recursive(account, all_emails)
    df = pl.DataFrame(all_emails)
    if SAVE_RAW_EXCEL:
        df.write_excel(RAW_EXCEL_FILE)
    pivot_df = create_pivot(df)
    print("\n" + "="*80)
    print("PIVOT TABLE (RTA team & TL)")
    print("="*80)
    print(pivot_df)
    print("="*80)
if __name__ == "__main__":
    main()

Scanned 2,000 items in Inbox...
   ✓ RE_ Final Schedules _ NL Chat Global OU Projections - WC 13th Apr'26\Global OU NL Chat 13th Apr'26.xlsx
   ✓ Final Schedules _ NL Chat Global OU Projections - WC 13th Apr'26\NL Chat Schedules - WC 13th Apr'26.xlsx
   ✓ Final Schedules _ NL Chat Global OU Projections - WC 13th Apr'26\NL Chat Schedules - WC 13th Apr'26.xlsx
   ✓ RE_ Final Schedules _ NL Chat Global OU Projections - WC 23rd Mar'26\Global OU NL Chat 23rd Mar'26.xlsx
   ✓ Final Schedules _ NL Chat Global OU Projections - WC 23rd Mar'26\Car Rental Training Slots 23rd Mar'26.xlsx
   ✓ Final Schedules _ NL Chat Global OU Projections - WC 23rd Mar'26\NL Chat Schedules - WC 23rd Mar'26.xlsx
   ✓ RE_ Final Schedules _ NL Chat Global OU Projections - WC 30th Mar'26\Global OU NL Chat 30th Mar'26 1.xlsx
   ✓ Final Schedules _ NL Chat Global OU Projections - WC 30th Mar'26\NL Chat Schedules - WC 30th Mar'26.xlsx
   ✓ NL Chat Global OU Projections and Final Schedules - WC 16th Mar'26\NL Chat Schedu

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13472\3404306264.py:148: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(index="Folder", columns="Date", values="Email Count", aggregate_function="sum")
